# TorchScript Model Inspector

This notebook provides tools to inspect and analyze saved TorchScript models, including:
- Loading and examining the model structure
- Printing the TorchScript code
- Analyzing the computation graph
- Testing model inference

In [21]:
import torch
import metatensor.torch
from metatensor.torch import TensorMap, Labels, TensorBlock
from metatensor.torch import Labels, TensorBlock, TensorMap

from metatomic.torch import (
    AtomisticModel,
    ModelCapabilities,
    ModelMetadata,
    ModelOutput,
    System,
    ModelEvaluationOptions,
)
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Load TorchScript Model

In [22]:
# Specify the path to your TorchScript model
model_path = "/mnt/projects/sne/Gohar/9.EPFL_visit/md_runs/test_ala2/md_single/metatomic.pt"  # Update this path

# Check if model exists
if os.path.exists(model_path):
    print(f"Loading TorchScript model from: {model_path}")
    model = torch.jit.load(model_path)
    print("Model loaded successfully!")
    print(f"Model type: {type(model)}")
else:
    print(f"Model not found at: {model_path}")
    print("Available model files in current directory:")
    for file in Path(".").rglob("*.pt"):
        print(f"  {file}")

Loading TorchScript model from: /mnt/projects/sne/Gohar/9.EPFL_visit/md_runs/test_ala2/md_single/metatomic.pt
Model loaded successfully!
Model type: <class 'torch.jit._script.RecursiveScriptModule'>


## 2. Print TorchScript Code

In [23]:
# Print the TorchScript code
if 'model' in locals():
    print("=" * 80)
    print("TORCHSCRIPT CODE:")
    print("=" * 80)
    print(model.code)
    print("=" * 80)
else:
    print("No model loaded. Please run the cell above first.")

TORCHSCRIPT CODE:
def forward(self,
    systems: List[__torch__.torch.classes.metatomic.System],
    options: __torch__.torch.classes.metatomic.ModelEvaluationOptions,
    check_consistency: bool) -> Dict[str, __torch__.torch.classes.metatensor.TensorMap]:
  _0 = __torch__.metatomic.torch.model._check_inputs
  _1 = "this model does not support the atomic type \'{}\' which is present in the input systems"
  _2 = __torch__.metatomic.torch.model._convert_systems_units
  _3 = __torch__.metatomic.torch.outputs._check_outputs
  _4 = "AtomisticModel::convert_units_output"
  _5 = "model produces values as \'{}\' for the \'{}\' output, but the engine requested \'{}\'"
  if check_consistency:
    _6 = __torch__.torch.autograd.profiler.record_function.__new__(__torch__.torch.autograd.profiler.record_function)
    _7 = (_6).__init__("AtomisticModel::check_inputs", None, )
    with _6:
      _capabilities = self._capabilities
      _requested_neighbor_lists = self._requested_neighbor_lists
      _m

## 3. Inspect Model Graph

In [24]:
# Print the computation graph
if 'model' in locals():
    print("=" * 80)
    print("COMPUTATION GRAPH:")
    print("=" * 80)
    print(model.graph)
    print("=" * 80)
else:
    print("No model loaded. Please run the loading cell first.")

COMPUTATION GRAPH:
graph(%self.1 : __torch__.metatomic.torch.model.AtomisticModel,
      %systems.1 : __torch__.torch.classes.metatomic.System[],
      %options.1 : __torch__.torch.classes.metatomic.ModelEvaluationOptions,
      %check_consistency.1 : bool):
  %220 : str = prim::Constant[value=""]() # /mnt/projects/sne/Gohar/9.EPFL_visit/.cenv/lib/python3.13/site-packages/metatomic/torch/model.py:450:40
  %182 : Function = prim::Constant[name="_check_outputs"]()
  %166 : str = prim::Constant[value="AtomisticModel::check_outputs"]() # /mnt/projects/sne/Gohar/9.EPFL_visit/.cenv/lib/python3.13/site-packages/metatomic/torch/model.py:436:33
  %147 : str = prim::Constant[value="Model::forward"]() # /mnt/projects/sne/Gohar/9.EPFL_visit/.cenv/lib/python3.13/site-packages/metatomic/torch/model.py:428:29
  %136 : Function = prim::Constant[name="_convert_systems_units"]()
  %123 : str = prim::Constant[value="length"]() # /mnt/projects/sne/Gohar/9.EPFL_visit/.cenv/lib/python3.13/site-packages/meta

## 4. Model Schema and Parameters

In [25]:
# Inspect model schema and parameters
if 'model' in locals():
    print("=" * 50)
    print("MODEL SCHEMA:")
    print("=" * 50)
    
    # Get the forward method schema
    try:
        schema = model.forward.schema
        print(f"Forward method schema: {schema}")
    except:
        print("Could not retrieve forward method schema")
    
    print("\n" + "=" * 50)
    print("MODEL PARAMETERS:")
    print("=" * 50)
    
    # List all parameters
    total_params = 0
    for name, param in model.named_parameters():
        param_count = param.numel()
        total_params += param_count
        print(f"{name}: {list(param.shape)} ({param_count:,} parameters)")
    
    print(f"\nTotal parameters: {total_params:,}")
    
    print("\n" + "=" * 50)
    print("MODEL BUFFERS:")
    print("=" * 50)
    
    # List all buffers (including normalization parameters)
    for name, buffer in model.named_buffers():
        print(f"{name}: {list(buffer.shape)} = {buffer}")
        
else:
    print("No model loaded. Please run the loading cell first.")

MODEL SCHEMA:
Forward method schema: forward(__torch__.metatomic.torch.model.AtomisticModel self, __torch__.torch.classes.metatomic.System[] systems, __torch__.torch.classes.metatomic.ModelEvaluationOptions options, bool check_consistency) -> Dict(str, __torch__.torch.classes.metatensor.TensorMap)

MODEL PARAMETERS:
module.model.encoder.encoder_hidden.0.weight: [256, 45] (11,520 parameters)
module.model.encoder.encoder_hidden.0.bias: [256] (256 parameters)
module.model.encoder.encoder_hidden.2.weight: [256] (256 parameters)
module.model.encoder.encoder_hidden.2.bias: [256] (256 parameters)
module.model.encoder.encoder_hidden.3.weight: [64, 256] (16,384 parameters)
module.model.encoder.encoder_hidden.3.bias: [64] (64 parameters)
module.model.encoder.encoder_hidden.5.weight: [64] (64 parameters)
module.model.encoder.encoder_hidden.5.bias: [64] (64 parameters)
module.model.encoder.encoder_mu.weight: [2, 64] (128 parameters)
module.model.encoder.encoder_mu.bias: [2] (2 parameters)
module.m

## 5. Test Model Inference

In [26]:
# Test the model with dummy input
if 'model' in locals():
    print("=" * 50)
    print("TESTING MODEL INFERENCE:")
    print("=" * 50)
    
    # Try to infer input shape from the model
    try:
        # Get input shape from the graph (this is a heuristic)
        # graph_str = str(model.graph)
        # print("Model graph preview:")
        # print(graph_str[:500] + "..." if len(graph_str) > 500 else graph_str)

        #alanine dipeptide system
        systems = [System(
            types=torch.tensor([1] * 10, dtype=torch.long),
            positions=torch.randn(10, 3, dtype=torch.float64),
            cell=torch.eye(3, dtype=torch.float64),
            pbc=torch.tensor([True, True, True]),
        )]
        systems_empty = [System(
            types=torch.tensor([1] * 0, dtype=torch.long),
            positions=torch.randn(0, 3, dtype=torch.float64),
            cell=torch.eye(3, dtype=torch.float64),
            pbc=torch.tensor([True, True, True]),
        )]
        options = ModelEvaluationOptions(
            length_unit="nanometer",
            outputs={"features": ModelOutput(quantity="", unit="none", per_atom=False),},
            selected_atoms=None,
        )
        
        
        # Run inference
        model.eval()
        with torch.no_grad():
            output = model(systems, options, False)
            
        print("Inference successful!")
        print(f"Output type: {type(output)}")
        print(f"Output keys: {list(output.keys())}")
        for key, tensor_map in output.items():
            print(f"Output '{key}': {tensor_map}")
        block = output["features"][0]
        print(f"Output block values: {block.values}")
        
    except Exception as e:
        print(f"Error during inference: {e}")
        print("\nTry adjusting the input_dim variable above to match your model's expected input size.")
        print("You can also check the model schema above for input requirements.")
        
else:
    print("No model loaded. Please run the loading cell first.")

TESTING MODEL INFERENCE:
Inference successful!
Output type: <class 'dict'>
Output keys: ['features']
Output 'features': TensorMap with 1 blocks
keys: _
      0
Output block values: tensor([[1.8669, 1.4202]], dtype=torch.float64)


## 6. Save TorchScript Code to File

In [18]:
# Save the TorchScript code to a text file for easier reading
if 'model' in locals():
    output_file = "torchscript_code.txt"
    
    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("TORCHSCRIPT CODE\n")
        f.write("=" * 80 + "\n\n")
        f.write(str(model.code))
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("COMPUTATION GRAPH\n")
        f.write("=" * 80 + "\n\n")
        f.write(str(model.graph))
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("MODEL PARAMETERS\n")
        f.write("=" * 80 + "\n\n")
        
        total_params = 0
        for name, param in model.named_parameters():
            param_count = param.numel()
            total_params += param_count
            f.write(f"{name}: {list(param.shape)} ({param_count:,} parameters)\n")
        f.write(f"\nTotal parameters: {total_params:,}\n")
        
        f.write("\n" + "=" * 80 + "\n")
        f.write("MODEL BUFFERS\n")
        f.write("=" * 80 + "\n\n")
        
        for name, buffer in model.named_buffers():
            f.write(f"{name}: {list(buffer.shape)} = {buffer}\n")
    
    print(f"TorchScript code and model info saved to: {output_file}")
else:
    print("No model loaded. Please run the loading cell first.")

TorchScript code and model info saved to: torchscript_code.txt


## 7. Advanced Analysis: Method Inspection

In [19]:
# Inspect all available methods in the TorchScript model
if 'model' in locals():
    print("=" * 50)
    print("AVAILABLE METHODS:")
    print("=" * 50)
    
    # Get all methods
    methods = [attr for attr in dir(model) if not attr.startswith('_')]
    print("Available methods:")
    for method in methods:
        try:
            attr = getattr(model, method)
            if callable(attr):
                print(f"  {method}()")
                # Try to get the schema if it's a TorchScript method
                try:
                    schema = attr.schema
                    print(f"    Schema: {schema}")
                except:
                    pass
            else:
                print(f"  {method} (attribute)")
        except:
            print(f"  {method} (could not inspect)")
    
    # Try to get the source code for specific methods
    print("\n" + "=" * 50)
    print("METHOD SOURCE CODE:")
    print("=" * 50)
    
    important_methods = ['forward', 'encoder', 'decoder']
    for method_name in important_methods:
        if hasattr(model, method_name):
            try:
                method = getattr(model, method_name)
                print(f"\n--- {method_name.upper()} METHOD ---")
                print(method.code)
            except Exception as e:
                print(f"\n--- {method_name.upper()} METHOD ---")
                print(f"Could not get source: {e}")
        else:
            print(f"\n--- {method_name.upper()} METHOD ---")
            print("Method not found")
            
else:
    print("No model loaded. Please run the loading cell first.")

AVAILABLE METHODS:
Available methods:
  T_destination (attribute)
  add_module()
  apply()
  bfloat16()
  buffers()
  call_super_init (attribute)
  children()
  code (attribute)
  code_with_constants (attribute)
  compile()
  cpu()
  cuda()
  define()
  double()
  dump_patches (attribute)
  eval()
  extra_repr()
  float()
  forward()
    Schema: forward(__torch__.metatomic.torch.model.AtomisticModel self, __torch__.torch.classes.metatomic.System[] systems, __torch__.torch.classes.metatomic.ModelEvaluationOptions options, bool check_consistency) -> Dict(str, __torch__.torch.classes.metatensor.TensorMap)
  forward()
    Schema: forward(__torch__.metatomic.torch.model.AtomisticModel self, __torch__.torch.classes.metatomic.System[] systems, __torch__.torch.classes.metatomic.ModelEvaluationOptions options, bool check_consistency) -> Dict(str, __torch__.torch.classes.metatensor.TensorMap)
  forward_magic_method()
  get_buffer()
  get_debug_state()
  get_extra_state()
  get_parameter()
  get_

## 8. Custom Input Testing

In [20]:
# Test with custom input dimensions
# Modify these values based on your specific model requirements

if 'model' in locals():
    print("=" * 50)
    print("CUSTOM INPUT TESTING:")
    print("=" * 50)
    
    # Customize these parameters for your model
    custom_input_dim = 64  # Change this to match your model's input dimension
    custom_batch_size = 3  # Test with multiple samples
    
    try:
        # Create custom input
        custom_input = torch.randn(custom_batch_size, custom_input_dim)
        print(f"Testing with custom input shape: {custom_input.shape}")
        
        # Run inference
        model.eval()
        with torch.no_grad():
            output = model(custom_input)
            
        print(f"\nInput statistics:")
        print(f"  Shape: {custom_input.shape}")
        print(f"  Mean: {custom_input.mean().item():.4f}")
        print(f"  Std: {custom_input.std().item():.4f}")
        print(f"  Min: {custom_input.min().item():.4f}")
        print(f"  Max: {custom_input.max().item():.4f}")
        
        print(f"\nOutput statistics:")
        print(f"  Shape: {output.shape}")
        print(f"  Mean: {output.mean().item():.4f}")
        print(f"  Std: {output.std().item():.4f}")
        print(f"  Min: {output.min().item():.4f}")
        print(f"  Max: {output.max().item():.4f}")
        
        # Show sample outputs
        print(f"\nSample outputs (first batch):")
        print(output[0])
        
        # Plot if output is reasonably sized
        if output.shape[1] <= 20:  # Only plot if latent dimension is small enough
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(custom_input[0].numpy(), 'b-', alpha=0.7, label='Input')
            plt.title('Sample Input')
            plt.xlabel('Dimension')
            plt.ylabel('Value')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            plt.subplot(1, 2, 2)
            plt.plot(output[0].numpy(), 'r-', alpha=0.7, label='Output')
            plt.title('Sample Output (Latent)')
            plt.xlabel('Latent Dimension')
            plt.ylabel('Value')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
    except Exception as e:
        print(f"Error during custom testing: {e}")
        print("\nTry adjusting the custom_input_dim variable above.")
        
else:
    print("No model loaded. Please run the loading cell first.")

CUSTOM INPUT TESTING:
Testing with custom input shape: torch.Size([3, 64])
Error during custom testing: forward() Expected a value of type 'List[__torch__.torch.classes.metatomic.System]' for argument 'systems' but instead found type 'Tensor'.
Position: 1
Value: tensor([[-1.8967, -0.1164,  1.2658,  1.8633, -0.2769, -1.1242,  0.2700,  0.9887,
         -0.5681, -0.8993,  0.5896,  0.3452,  0.1474,  0.2525,  0.6889,  0.3270,
          1.5420,  0.1554,  0.4060, -0.6508, -1.1791,  0.9682, -0.7274,  1.1232,
          1.1643,  0.7338, -0.4056,  0.7132,  0.6716,  0.0505,  0.1316, -0.4957,
         -0.9451,  1.3750,  0.5463, -0.7697, -1.4100,  0.2681, -1.1308,  0.7186,
          0.1514,  1.3134,  0.1810,  1.3779,  0.7631,  1.0280,  0.1762, -0.3870,
         -0.4952,  1.1825, -0.0045,  1.7053,  0.0953,  0.0081, -1.2235,  0.0654,
          0.6223, -0.2878,  0.2035, -1.6953,  2.0271,  1.6403, -1.2467, -1.0537],
        [ 0.2978,  0.5468,  0.7166,  0.9345,  0.0559, -0.1707,  1.7529,  0.1369,
       

## Usage Instructions

1. **Update the model path** in the first code cell to point to your TorchScript model file
2. **Run all cells** to get a comprehensive analysis of your model
3. **Adjust input dimensions** in the testing cells if needed
4. **Check the generated text file** for a complete report

This notebook will help you understand:
- The actual compiled TorchScript code
- Model architecture and parameters
- Input/output shapes and behavior
- Any potential issues with the compilation